# 📷 Real-Time Student Attention Monitor
**MediaPipe Face Mesh + Random Forest → Per-Face Label Overlay**

---
### Labels
| ID | Label | Signal |
|---|---|---|
| 0 | 🟢 Attentive | High EAR, low yaw, centered face |
| 1 | 🟡 Distracted | High yaw / nose offset (looking away) |
| 2 | 🟠 Drowsy | Low EAR, high MAR (eyes closing, yawning) |
| 3 | 🔴 Absent | Very low EAR (eyes fully closed) |

> **Press `q`** in the webcam window to stop.

## 1. 📦 Install Dependencies

In [1]:
import sys
!{sys.executable} -m pip install "mediapipe>=0.10" opencv-python scikit-learn pandas numpy --quiet
print('✅ Dependencies installed!')

✅ Dependencies installed!


## 2. 📦 Import Libraries & Detect MediaPipe Version

In [2]:
import cv2
import numpy as np
import pandas as pd
import time
import warnings
warnings.filterwarnings('ignore')

import mediapipe as mp
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

mp_version = tuple(int(x) for x in mp.__version__.split('.')[:2])
NEW_MP_API = mp_version >= (0, 10)

print(f'MediaPipe version : {mp.__version__}')
print(f'Using new API     : {NEW_MP_API}')
print('✅ Libraries imported!')

MediaPipe version : 0.10.32
Using new API     : True
✅ Libraries imported!


## 3. 🤖 Train Model on Dataset

In [3]:
def train_model(csv_path='/Users/ashwin/Documents/MTech Project/Ashwin BS Final/attention_dataset (1).csv'):
    print('[INFO] Loading dataset and training model...')
    df = pd.read_csv(csv_path)

    feature_cols = [
        'left_ear', 'right_ear', 'avg_ear', 'mar',
        'yaw_deg', 'pitch_deg', 'roll_deg',
        'ear_asymmetry', 'nose_centrality'
    ]

    df['ear_mar_ratio']        = df['avg_ear'] / (df['mar'] + 1e-6)
    df['head_angle_magnitude'] = np.sqrt(df['yaw_deg']**2 + df['pitch_deg']**2 + df['roll_deg']**2)
    df['total_distraction']    = df['nose_centrality'] + df['ear_asymmetry'] + np.abs(df['yaw_deg']) / 90

    all_features = feature_cols + ['ear_mar_ratio', 'head_angle_magnitude', 'total_distraction']

    X = df[all_features]
    y = df['label_id']

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)

    acc = model.score(X_test, y_test)
    print(f'[INFO] ✅ Model trained! Test Accuracy: {acc:.4f}')
    return model, all_features


model, all_features = train_model(csv_path='/Users/ashwin/Documents/MTech Project/Ashwin BS Project/Ashwin BS Final/attention_dataset (1).csv')

[INFO] Loading dataset and training model...
[INFO] ✅ Model trained! Test Accuracy: 0.9969


## 4. 🧮 Facial Landmark Utility Functions

In [4]:
LEFT_EYE  = [362, 385, 387, 263, 373, 380]
RIGHT_EYE = [33,  160, 158, 133, 153, 144]
NOSE_TIP         = 1
FACE_LEFT_BOUND  = 234
FACE_RIGHT_BOUND = 454

# Boundary landmarks used to compute face bounding box
FACE_BBOX_POINTS = [10, 152, 234, 454]   # top, bottom, left, right


def euclidean(p1, p2):
    return np.linalg.norm(np.array(p1) - np.array(p2))


def get_landmark_px(lm, idx, w, h):
    pt = lm[idx]
    return (pt.x * w, pt.y * h)


def get_face_bbox(lm, w, h, padding=20):
    """
    Compute a tight bounding box around the face from all landmarks.
    Returns (x1, y1, x2, y2) in pixel coordinates.
    """
    xs = [lm[i].x * w for i in range(len(lm))]
    ys = [lm[i].y * h for i in range(len(lm))]
    x1 = max(0,     int(min(xs)) - padding)
    y1 = max(0,     int(min(ys)) - padding)
    x2 = min(w - 1, int(max(xs)) + padding)
    y2 = min(h - 1, int(max(ys)) + padding)
    return x1, y1, x2, y2


def compute_ear(lm, eye_indices, w, h):
    pts = [get_landmark_px(lm, i, w, h) for i in eye_indices]
    v1 = euclidean(pts[1], pts[5])
    v2 = euclidean(pts[2], pts[4])
    hd = euclidean(pts[0], pts[3])
    return (v1 + v2) / (2.0 * hd + 1e-6)


def compute_mar(lm, w, h):
    top   = get_landmark_px(lm, 13,  w, h)
    bot   = get_landmark_px(lm, 14,  w, h)
    left  = get_landmark_px(lm, 61,  w, h)
    right = get_landmark_px(lm, 291, w, h)
    return euclidean(top, bot) / (euclidean(left, right) + 1e-6)


def compute_head_pose(lm, w, h):
    nose  = np.array(get_landmark_px(lm, NOSE_TIP, w, h))
    l_eye = np.array(get_landmark_px(lm, 33,  w, h))
    r_eye = np.array(get_landmark_px(lm, 263, w, h))
    mouth = np.array(get_landmark_px(lm, 13,  w, h))
    eye_mid   = (l_eye + r_eye) / 2
    yaw_deg   = (nose[0] - eye_mid[0]) / (w / 90.0)
    pitch_deg = (nose[1] - (eye_mid[1] + mouth[1]) / 2) / (h / 90.0)
    delta     = r_eye - l_eye
    roll_deg  = float(np.degrees(np.arctan2(delta[1], delta[0])))
    return float(yaw_deg), float(pitch_deg), roll_deg


def compute_nose_centrality(lm, w, h):
    nose_x  = lm[NOSE_TIP].x * w
    left_x  = lm[FACE_LEFT_BOUND].x  * w
    right_x = lm[FACE_RIGHT_BOUND].x * w
    center  = (left_x + right_x) / 2
    width   = right_x - left_x + 1e-6
    return abs(nose_x - center) / width


def extract_features(lm, w, h):
    """Extract all model features from a single face landmark set."""
    left_ear  = compute_ear(lm, LEFT_EYE,  w, h)
    right_ear = compute_ear(lm, RIGHT_EYE, w, h)
    avg_ear   = (left_ear + right_ear) / 2
    mar       = compute_mar(lm, w, h)
    yaw, pitch, roll = compute_head_pose(lm, w, h)
    ear_asym  = abs(left_ear - right_ear)
    nose_cent = compute_nose_centrality(lm, w, h)
    return {
        'left_ear': left_ear, 'right_ear': right_ear, 'avg_ear': avg_ear,
        'mar': mar, 'yaw_deg': yaw, 'pitch_deg': pitch, 'roll_deg': roll,
        'ear_asymmetry': ear_asym, 'nose_centrality': nose_cent,
        'ear_mar_ratio':        avg_ear / (mar + 1e-6),
        'head_angle_magnitude': np.sqrt(yaw**2 + pitch**2 + roll**2),
        'total_distraction':    nose_cent + ear_asym + abs(yaw) / 90,
    }


print('✅ Landmark functions defined!')

✅ Landmark functions defined!


## 5. 🎨 Per-Face Drawing Functions

In [5]:
LABEL_INFO = {
    0: ('Attentive',  (39,  174,  96)),
    1: ('Distracted', (243, 156,  18)),
    2: ('Drowsy',     (211,  84,   0)),
    3: ('Absent',     (192,  57,  43)),
}


def draw_face_annotation(frame, bbox, label_id, proba):
    """
    Draw bounding box + label tag ABOVE each detected face.

    Layout:
      ┌──────────────────┐  ← label pill above the box
      │  Attentive  94%  │
      └──────────────────┘
      ┌──────────────────┐
      │                  │  ← face bounding box
      │      face        │
      │                  │
      └──────────────────┘
    """
    x1, y1, x2, y2 = bbox
    label_name, color = LABEL_INFO[label_id]
    confidence = proba[label_id]

    # ── Face bounding box ────────────────────────────────────────────────────
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

    # ── Label pill above the bounding box ───────────────────────────────────
    text       = f'{label_name}  {confidence:.0%}'
    font       = cv2.FONT_HERSHEY_DUPLEX
    font_scale = 0.65
    thickness  = 1
    (tw, th), baseline = cv2.getTextSize(text, font, font_scale, thickness)

    pill_x1 = x1
    pill_y2 = y1 - 4                          # just above the box
    pill_y1 = pill_y2 - th - baseline - 8
    pill_x2 = x1 + tw + 14

    # Filled pill background
    cv2.rectangle(frame, (pill_x1, pill_y1), (pill_x2, pill_y2), color, -1)
    # White text on pill
    cv2.putText(frame, text,
                (pill_x1 + 7, pill_y2 - baseline - 2),
                font, font_scale, (255, 255, 255), thickness, cv2.LINE_AA)

    # ── Mini probability bar strip below the label ───────────────────────────
    bar_top = y2 + 4
    bar_h   = 6
    face_w  = x2 - x1
    for lid, (_, lcolor) in LABEL_INFO.items():
        seg_w = int(proba[lid] * face_w)
        offset = int(sum(proba[i] for i in range(lid)) * face_w)
        cv2.rectangle(frame,
                      (x1 + offset, bar_top),
                      (x1 + offset + seg_w, bar_top + bar_h),
                      lcolor, -1)
    # bar border
    cv2.rectangle(frame, (x1, bar_top), (x2, bar_top + bar_h), (80, 80, 80), 1)

    return frame


def draw_hud(frame, fps, n_faces):
    """Draw a minimal HUD: FPS and face count in the top-right corner."""
    h, w = frame.shape[:2]
    hud_text = f'FPS: {fps:.1f}   Faces: {n_faces}'
    cv2.putText(frame, hud_text, (w - 220, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.65, (200, 200, 200), 1, cv2.LINE_AA)
    return frame


print('✅ Per-face drawing functions defined!')

✅ Per-face drawing functions defined!


## 6. 🔧 Initialize MediaPipe (Version-Compatible, Multi-Face)

In [6]:
MAX_FACES = 6   # maximum number of faces to track simultaneously

if NEW_MP_API:
    from mediapipe.tasks import python as mp_python
    from mediapipe.tasks.python.vision import FaceLandmarker, FaceLandmarkerOptions, RunningMode
    import urllib.request, os

    MODEL_PATH = 'face_landmarker.task'
    if not os.path.exists(MODEL_PATH):
        print('[INFO] Downloading face_landmarker.task (~3 MB)...')
        url = ('https://storage.googleapis.com/mediapipe-models/'
               'face_landmarker/face_landmarker/float16/1/face_landmarker.task')
        urllib.request.urlretrieve(url, MODEL_PATH)
        print('[INFO] Download complete.')

    _options = FaceLandmarkerOptions(
        base_options=mp_python.BaseOptions(model_asset_path=MODEL_PATH),
        running_mode=RunningMode.IMAGE,
        num_faces=MAX_FACES,                  # ← multi-face
        min_face_detection_confidence=0.5,
        min_face_presence_confidence=0.5,
        min_tracking_confidence=0.5,
    )
    face_landmarker = FaceLandmarker.create_from_options(_options)
    print(f'[INFO] MediaPipe >= 0.10 Tasks API — max {MAX_FACES} faces.')

    def get_all_landmarks(frame_rgb):
        """Return list of landmark sets (one per detected face)."""
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
        result   = face_landmarker.detect(mp_image)
        return result.face_landmarks   # list of lists

else:
    _face_mesh = mp.solutions.face_mesh.FaceMesh(
        max_num_faces=MAX_FACES,              # ← multi-face
        refine_landmarks=True,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5,
    )
    print(f'[INFO] MediaPipe < 0.10 solutions API — max {MAX_FACES} faces.')

    def get_all_landmarks(frame_rgb):
        result = _face_mesh.process(frame_rgb)
        if result.multi_face_landmarks:
            return [f.landmark for f in result.multi_face_landmarks]
        return []


print('✅ get_all_landmarks() ready.')

[INFO] MediaPipe >= 0.10 Tasks API — max 6 faces.
✅ get_all_landmarks() ready.


W0000 00:00:1775537340.693430 2990464 face_landmarker_graph.cc:180] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
I0000 00:00:1775537340.797410 2990464 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4 Pro
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1775537340.803045 2990466 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1775537340.812446 2990465 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


## 7. 🎥 Real-Time Webcam Loop — Label Appears Above Each Face

> **Press `q`** to quit. Change `CAMERA_INDEX = 1` for an external camera.  
> Change `MAX_FACES` in Cell 6 to track more students at once.

In [7]:
CAMERA_INDEX = 0       # 0 = built-in webcam, 1 = USB camera
SMOOTH_ALPHA = 0.35    # prediction smoothing per face
FRAME_WIDTH  = 640
FRAME_HEIGHT = 480

cap = cv2.VideoCapture(CAMERA_INDEX)
if not cap.isOpened():
    raise RuntimeError(f'Cannot open camera at index {CAMERA_INDEX}. Try CAMERA_INDEX = 1.')

cap.set(cv2.CAP_PROP_FRAME_WIDTH,  FRAME_WIDTH)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, FRAME_HEIGHT)

print('[INFO] Webcam opened. Press "q" to quit.')

# Per-face smoothed probability buffers keyed by face index
# (resets when face count changes — simple but effective)
face_smooth = {}

prev_time = time.time()

while True:
    ret, frame = cap.read()
    if not ret:
        continue

    frame = cv2.flip(frame, 1)
    h, w  = frame.shape[:2]
    rgb   = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    curr_time = time.time()
    fps       = 1.0 / (curr_time - prev_time + 1e-6)
    prev_time = curr_time

    all_lm = get_all_landmarks(rgb)   # list of landmark sets
    n_faces = len(all_lm)

    # Clean up smoothing buffers for faces that disappeared
    for key in list(face_smooth.keys()):
        if key >= n_faces:
            del face_smooth[key]

    for face_idx, lm in enumerate(all_lm):
        # ── Compute bounding box ─────────────────────────────────────────────
        bbox = get_face_bbox(lm, w, h, padding=20)

        # ── Extract features & predict ───────────────────────────────────────
        feat_dict = extract_features(lm, w, h)
        X_live    = pd.DataFrame([feat_dict])[all_features]
        proba     = model.predict_proba(X_live)[0]

        # ── Per-face exponential smoothing ───────────────────────────────────
        if face_idx not in face_smooth:
            face_smooth[face_idx] = np.ones(4) / 4
        face_smooth[face_idx] = (
            SMOOTH_ALPHA * proba + (1 - SMOOTH_ALPHA) * face_smooth[face_idx]
        )
        smoothed = face_smooth[face_idx]
        label_id = int(np.argmax(smoothed))

        # ── Draw label above this face ───────────────────────────────────────
        frame = draw_face_annotation(frame, bbox, label_id, smoothed)

    # ── HUD: FPS + face count ────────────────────────────────────────────────
    draw_hud(frame, fps, n_faces)

    if n_faces == 0:
        cv2.putText(frame, 'No face detected', (20, 50),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 255), 2, cv2.LINE_AA)

    cv2.imshow('Student Attention Monitor', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        print('[INFO] Quit signal received.')
        break

cap.release()
cv2.destroyAllWindows()
print('[INFO] ✅ Session ended.')

[INFO] Webcam opened. Press "q" to quit.
[INFO] Quit signal received.
[INFO] ✅ Session ended.


## 8. 📝 Troubleshooting

| Problem | Fix |
|---|---|
| `module 'mediapipe' has no attribute 'solutions'` | ✅ Auto-detected — uses new Tasks API |
| `face_landmarker.task not found` | Cell 6 downloads it automatically |
| Window doesn't open | Run as `.py` file outside Jupyter |
| Camera not found | Change `CAMERA_INDEX = 1` or `2` |
| Want to track more students | Increase `MAX_FACES` in Cell 6 |
| Low FPS with many faces | Reduce `n_estimators=50` in Cell 3 |